# RF v2 — Ground Truth vs Prediction: Patch Comparison

Runs the trained RF v2 model over the **geographic test band (rows 9216–12288)**, then automatically selects:
- One **well-classified** 50×50 patch (highest mean wetland F1, must contain ≥2 classes with wetlands dominant)
- One **poorly-classified** 50×50 patch (lowest mean wetland F1, same constraints)

Output: 4-panel figure saved to Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q rasterio joblib

In [ ]:
import os, shutil
from pathlib import Path

DRIVE_BASE_SRC = '/content/drive/MyDrive/EarthEngine'
LOCAL_BASE     = '/content/local_tiles'

if not os.path.exists(LOCAL_BASE):
    print('Copying tiles to local disk (one-time, ~5–10 min)...')
    os.makedirs(LOCAL_BASE, exist_ok=True)
    tile_files = list(Path(DRIVE_BASE_SRC).glob('*.tif'))
    for tf in tile_files:
        dst = os.path.join(LOCAL_BASE, tf.name)
        if not os.path.exists(dst):
            shutil.copy2(str(tf), dst)
    for fname in [
        'bow_river_wetlands_10m_final.tif',
        'rf_wetland_model_middle_v2_20260310_124941.pkl',
        'rf_scaler_middle_v2_20260310_124941.pkl',
    ]:
        src = os.path.join(DRIVE_BASE_SRC, fname)
        dst = os.path.join(LOCAL_BASE, fname)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)
    print(f'Done. {len(list(Path(LOCAL_BASE).iterdir()))} files copied.')
else:
    print(f'Local tiles already present ({len(list(Path(LOCAL_BASE).iterdir()))} files).')

In [ ]:
import os
import numpy as np
import joblib
import rasterio
from rasterio.windows import Window
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams.update({"font.family": "DejaVu Sans"})
print("Imports OK")

## Configuration
Edit the paths below if your Drive layout differs.

In [ ]:
LOCAL_BASE     = '/content/local_tiles'

LABELS_PATH    = os.path.join(LOCAL_BASE, 'bow_river_wetlands_10m_final.tif')
MODEL_PATH     = os.path.join(LOCAL_BASE, 'rf_wetland_model_middle_v2_20260310_124941.pkl')
SCALER_PATH    = os.path.join(LOCAL_BASE, 'rf_scaler_middle_v2_20260310_124941.pkl')
EMBEDDINGS_DIR = LOCAL_BASE
OUTPUT_PATH    = '/content/drive/MyDrive/rf_patch_comparison.png'

# Test band (from model metadata)
TEST_ROW_MIN = 9216
TEST_ROW_MAX = 12288

# Patch settings
PATCH_SIZE          = 50     # 50×50 px = 500 m × 500 m at 10 m/px
MIN_CLASSES         = 2      # min unique GT classes in patch
MIN_WETLAND_CLASSES = 1      # at least 1 wetland class (1–5) required
MIN_VALID_COVERAGE  = 0.85   # fraction of patch needing valid predictions
MAX_BG_FRACTION     = 0.60   # reject patches where background > this fraction of GT

# Rows processed per chunk inside each tile — keeps peak RAM ~200 MB
ROW_CHUNK = 256

NODATA = 255

for p in [LABELS_PATH, MODEL_PATH, SCALER_PATH, EMBEDDINGS_DIR]:
    assert os.path.exists(p), f'Not found: {p}'
print('All paths verified ✓')

## Class Definitions

In [ ]:
CLASS_NAMES = {
    0: 'Background',
    1: 'Fen (Graminoid)',
    2: 'Fen (Woody)',
    3: 'Marsh',
    4: 'Shallow Open Water',
    5: 'Swamp',
}

CLASS_COLORS = np.array([
    [0.75, 0.75, 0.75],   # 0 Background
    [0.85, 0.75, 0.30],   # 1 Fen (Graminoid)
    [0.40, 0.65, 0.25],   # 2 Fen (Woody)
    [0.20, 0.55, 0.20],   # 3 Marsh
    [0.20, 0.55, 0.90],   # 4 Shallow Open Water
    [0.00, 0.30, 0.55],   # 5 Swamp
], dtype=np.float32)

def labels_to_rgb(arr):
    rgb = np.zeros((*arr.shape, 3), dtype=np.float32)
    for cls, color in enumerate(CLASS_COLORS):
        rgb[arr == cls] = color
    rgb[arr == NODATA] = [0.12, 0.12, 0.12]
    return rgb

print('Class definitions ready ✓')

## Step 1 — Load Model and Scaler

In [ ]:
rf_model = joblib.load(MODEL_PATH)
scaler   = joblib.load(SCALER_PATH)
print(f'Model:  {os.path.basename(MODEL_PATH)}  ({rf_model.n_estimators} trees, {rf_model.n_features_in_} features)')
print(f'Scaler: {os.path.basename(SCALER_PATH)}')

## Step 2 — Load Ground Truth Test Band

In [ ]:
with rasterio.open(LABELS_PATH) as src:
    FULL_HEIGHT = src.height
    FULL_WIDTH  = src.width

BAND_HEIGHT = TEST_ROW_MAX - TEST_ROW_MIN
print(f'Full raster: {FULL_HEIGHT} × {FULL_WIDTH}')
print(f'Test band:   rows {TEST_ROW_MIN}–{TEST_ROW_MAX}  ({BAND_HEIGHT} rows × {FULL_WIDTH} cols)')

print('\nLoading ground truth for test band...')
with rasterio.open(LABELS_PATH) as src:
    ground_truth = src.read(1, window=Window(0, TEST_ROW_MIN, FULL_WIDTH, BAND_HEIGHT)).astype(np.int16)

print(f'Ground truth shape: {ground_truth.shape}')
for cls in range(6):
    n = (ground_truth == cls).sum()
    print(f'  Class {cls} ({CLASS_NAMES[cls]:20s}): {n:>8,}  ({100*n/ground_truth.size:.1f}%)')

## Step 3 — Discover Test Tiles and Run Inference
This may take a few minutes depending on the number of tiles.

In [ ]:
all_tiles  = sorted(Path(EMBEDDINGS_DIR).glob('*.tif'))
test_tiles = []
for tf in all_tiles:
    parts = tf.stem.split('-')
    if len(parts) >= 3:
        try:
            row_off = int(parts[-2])
            if TEST_ROW_MIN <= row_off <= TEST_ROW_MAX:
                test_tiles.append(tf)
        except ValueError:
            pass

print(f'Total tiles in directory: {len(all_tiles)}')
print(f'Test-band tiles:          {len(test_tiles)}')
if not test_tiles:
    raise RuntimeError('No test tiles found. Check EMBEDDINGS_DIR and tile naming (*-ROW-COL.tif).')

In [ ]:
predictions = np.full((BAND_HEIGHT, FULL_WIDTH), NODATA, dtype=np.uint8)

for tile_path in tqdm(test_tiles, desc='Inferring tiles'):
    parts = tile_path.stem.split('-')
    try:
        tile_row_off = int(parts[-2])
        tile_col_off = int(parts[-1])
    except (ValueError, IndexError):
        continue

    with rasterio.open(tile_path) as tile_src:
        if tile_src.count != 64:
            continue
        tile_h  = tile_src.height
        tile_w  = tile_src.width
        valid_w = min(tile_w, FULL_WIDTH - tile_col_off)

        clip_r0 = max(tile_row_off, TEST_ROW_MIN)
        clip_r1 = min(tile_row_off + tile_h, TEST_ROW_MIN + BAND_HEIGHT)
        if clip_r0 >= clip_r1:
            continue

        # Process in row chunks to cap peak RAM usage
        row_cursor = clip_r0
        while row_cursor < clip_r1:
            chunk_end = min(row_cursor + ROW_CHUNK, clip_r1)
            chunk_h   = chunk_end - row_cursor
            local_r0  = row_cursor - tile_row_off

            chunk_data = tile_src.read(window=Window(0, local_r0, valid_w, chunk_h))  # (64, chunk_h, valid_w)
            n_px       = chunk_h * valid_w
            pixels     = chunk_data.reshape(64, n_px).T.astype(np.float32)
            del chunk_data  # free immediately

            nan_mask   = ~np.isnan(pixels).any(axis=1)
            preds_flat = np.full(n_px, NODATA, dtype=np.uint8)
            if nan_mask.any():
                # Cast back to float32 — sklearn returns float64 by default, doubling RAM
                X_scaled = scaler.transform(pixels[nan_mask]).astype(np.float32)
                preds_flat[nan_mask] = rf_model.predict(X_scaled).astype(np.uint8)
                del X_scaled
            del pixels

            out_r0 = row_cursor - TEST_ROW_MIN
            predictions[out_r0:out_r0 + chunk_h, tile_col_off:tile_col_off + valid_w] = preds_flat.reshape(chunk_h, valid_w)
            row_cursor = chunk_end

valid = predictions != NODATA
acc_overall = (predictions[valid] == ground_truth[valid]).mean()
print(f'\nOverall test-band accuracy: {acc_overall*100:.2f}%  ({valid.sum():,} valid pixels)')

## Step 4 — Find Best and Worst Patches

In [ ]:
from sklearn.metrics import f1_score

def patch_mean_wetland_f1(gt_valid, pred_valid):
    """Mean F1 across wetland classes (1–5) present in this patch."""
    classes_present = [c for c in np.unique(gt_valid) if 1 <= c <= 5]
    if not classes_present:
        return 0.0
    all_classes = list(np.unique(np.concatenate([gt_valid, pred_valid])))
    scores = f1_score(gt_valid, pred_valid, labels=all_classes, average=None, zero_division=0)
    label_to_score = dict(zip(all_classes, scores))
    return float(np.mean([label_to_score.get(c, 0.0) for c in classes_present]))

print(f'Scanning {PATCH_SIZE}×{PATCH_SIZE} patches...')
candidates = []

for pr in range(BAND_HEIGHT // PATCH_SIZE):
    r0 = pr * PATCH_SIZE
    for pc in range(FULL_WIDTH // PATCH_SIZE):
        c0 = pc * PATCH_SIZE
        gt_p   = ground_truth[r0:r0+PATCH_SIZE, c0:c0+PATCH_SIZE]
        pred_p = predictions [r0:r0+PATCH_SIZE, c0:c0+PATCH_SIZE]

        valid_mask = pred_p != NODATA
        if valid_mask.sum() / (PATCH_SIZE * PATCH_SIZE) < MIN_VALID_COVERAGE:
            continue

        gt_valid   = gt_p[valid_mask]
        pred_valid = pred_p[valid_mask]
        unique_cls = np.unique(gt_valid)

        # Must have ≥ MIN_CLASSES unique GT classes
        if len(unique_cls) < MIN_CLASSES:
            continue

        # Must include at least one wetland class
        if not np.any((unique_cls >= 1) & (unique_cls <= 5)):
            continue

        # Reject background-dominated patches
        bg_frac = (gt_valid == 0).sum() / len(gt_valid)
        if bg_frac > MAX_BG_FRACTION:
            continue

        score = patch_mean_wetland_f1(gt_valid, pred_valid)
        candidates.append((score, r0, c0))

print(f'Valid candidate patches: {len(candidates)}')
if len(candidates) < 2:
    raise RuntimeError('Not enough valid patches. Try increasing MAX_BG_FRACTION or reducing MIN_VALID_COVERAGE.')

candidates.sort(key=lambda x: x[0])
worst_f1, worst_r, worst_c = candidates[0]
best_f1,  best_r,  best_c  = candidates[-1]

print(f'\nBest  patch @ row={best_r  + TEST_ROW_MIN}, col={best_c}  — mean wetland F1={best_f1:.3f}')
print(f'Worst patch @ row={worst_r + TEST_ROW_MIN}, col={worst_c} — mean wetland F1={worst_f1:.3f}')

for label, r0, c0 in [('Best', best_r, best_c), ('Worst', worst_r, worst_c)]:
    gt_p = ground_truth[r0:r0+PATCH_SIZE, c0:c0+PATCH_SIZE]
    print(f'\n  {label} patch GT classes:')
    for cls in np.unique(gt_p):
        n = (gt_p == cls).sum()
        print(f'    Class {cls} ({CLASS_NAMES.get(int(cls), "?"):20s}): {n:,} px  ({100*n/gt_p.size:.1f}%)')

## Step 5 — Render and Save 4-Panel Figure

In [ ]:
def extract(arr, r0, c0):
    return arr[r0:r0+PATCH_SIZE, c0:c0+PATCH_SIZE]

best_gt    = extract(ground_truth, best_r,  best_c)
best_pred  = extract(predictions,  best_r,  best_c)
worst_gt   = extract(ground_truth, worst_r, worst_c)
worst_pred = extract(predictions,  worst_r, worst_c)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
fig.patch.set_facecolor('white')

for ax, rgb, title in [
    (axes[0, 0], labels_to_rgb(best_gt),    'Ground Truth'),
    (axes[0, 1], labels_to_rgb(best_pred),  'RF Prediction'),
    (axes[1, 0], labels_to_rgb(worst_gt),   'Ground Truth'),
    (axes[1, 1], labels_to_rgb(worst_pred), 'RF Prediction'),
]:
    ax.imshow(rgb, interpolation='nearest')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=6)
    ax.axis('off')

for ax, label, score in [
    (axes[0, 0], 'Well-classified region',   best_f1),
    (axes[1, 0], 'Poorly-classified region', worst_f1),
]:
    ax.set_ylabel(
        f'{label}\n(mean wetland F1: {score:.3f})',
        fontsize=10, labelpad=8, color='#2c3e50', fontweight='bold'
    )

legend_patches = [
    mpatches.Patch(facecolor=CLASS_COLORS[c], edgecolor='#888888',
                   linewidth=0.5, label=f'{c}: {CLASS_NAMES[c]}')
    for c in range(6)
]
fig.legend(
    handles=legend_patches, loc='lower center', ncol=3, fontsize=9,
    bbox_to_anchor=(0.5, 0.01), framealpha=0.9, edgecolor='#CCCCCC',
)

fig.suptitle(
    'RF v2 — Ground Truth vs Prediction: Test Band Patch Comparison\n'
    f'(Test region: rows {TEST_ROW_MIN}–{TEST_ROW_MAX},  patch size: {PATCH_SIZE}×{PATCH_SIZE} px @ 10 m/px)',
    fontsize=13, fontweight='bold', y=0.98
)

plt.tight_layout(rect=[0, 0.10, 1, 0.96])
plt.savefig(OUTPUT_PATH, dpi=200, bbox_inches='tight', facecolor='white')
print(f'✅  Saved: {OUTPUT_PATH}')
plt.show()